<a href="https://colab.research.google.com/github/krishna2426/governance-agent/blob/main/governance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q nest_asyncio crewai pydantic PyPDF2 joblib langchain-google-genai

In [ ]:
import os
import pickle
import PyPDF2
import json
from pydantic import BaseModel
from typing import List, Dict
from google.colab import userdata, files
from crewai import Agent, Task, Crew, LLM

# ==========================================
# 0. SHARED CONTRACT SCHEMA (slide 9)
# Share this class with your teammates — it's
# the agreed shape every agent's output must match.
# ==========================================
class AgentReport(BaseModel):
    agent_name: str
    score: int
    status: str
    findings: List[str]
    recommendations: List[str]
    metadata: Dict = {}

# ==========================================
# 1. UPLOAD REAL MODEL + DOCS
# ==========================================
print("Upload your model (.pkl) file:")
uploaded_model = files.upload()
UPLOADED_PKL = list(uploaded_model.keys())[0]

print("Upload your documentation (.pdf/.md/.txt) file:")
uploaded_docs = files.upload()
if len(uploaded_docs) == 0:
    print("⚠️ No documentation uploaded — proceeding with placeholder.")
    UPLOADED_DOCS = "no_docs.txt"
    with open(UPLOADED_DOCS, "w") as f:
        f.write("No formal documentation was provided for this model.")
else:
    UPLOADED_DOCS = list(uploaded_docs.keys())[0]
    print(f"Got: {UPLOADED_DOCS}")

# ==========================================
# 2. ARTIFACT EXTRACTION NODE
# ==========================================
def extract_documentation(file_path):
    print(f"Extracting docs from: {file_path}...")
    _, ext = os.path.splitext(file_path)
    extracted_text = ""
    try:
        if ext.lower() == '.pdf':
            with open(file_path, 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                for page in reader.pages:
                    extracted_text += page.extract_text() + "\n"
        elif ext.lower() in ['.md', '.txt']:
            with open(file_path, 'r', encoding='utf-8') as file:
                extracted_text = file.read()
        return extracted_text.strip()
    except Exception as e:
        return f"Error reading documentation: {str(e)}"

def extract_model_artifacts(pkl_path):
    print(f"Extracting metadata from: {pkl_path}...")
    try:
        with open(pkl_path, 'rb') as file:
            model_package = pickle.load(file)

        if isinstance(model_package, dict):
            metadata = model_package.get("metadata", {})
            feature_schema = model_package.get("feature_schema", {})
            return metadata, feature_schema

        model = model_package
        metadata = {
            "model_type": type(model).__name__,
            "module": type(model).__module__,
        }

        # Sklearn-style wrapper (XGBClassifier / XGBRegressor)
        if hasattr(model, "get_params"):
            metadata["hyperparameters"] = model.get_params()

        # Native xgboost.Booster (xgb.train API)
        elif hasattr(model, "save_config"):
            metadata["booster_config"] = json.loads(model.save_config())

        feature_schema = {}
        if hasattr(model, "feature_names_in_"):
            feature_schema = {f: "unknown" for f in model.feature_names_in_}
        elif hasattr(model, "feature_names") and model.feature_names:
            feature_schema = {f: "unknown" for f in model.feature_names}
        elif hasattr(model, "n_features_in_"):
            feature_schema = {"n_features": model.n_features_in_}

        if not feature_schema:
            feature_schema = {"warning": "Could not auto-detect feature schema from model object — relying on documentation only"}

        return metadata, feature_schema

    except Exception as e:
        return {"error": str(e)}, {"error": str(e)}

# ==========================================
# 3. INITIALIZE GEMINI AGENT
# ==========================================
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

governance_llm = LLM(
    model="gemini/gemini-2.5-flash",
    api_key=os.environ["GEMINI_API_KEY"],
    temperature=0.1
)

governance_agent = Agent(
    role='Lead AI Governance & Compliance Auditor',
    goal='Strictly validate regulatory and corporate governance compliance across model artifacts.',
    backstory=(
        "You are an expert AI governance auditor specializing in frameworks like SR 11-7 and the EU AI Act. "
        "Your job is to inspect model documentation, feature metadata, and explainability reports to ensure "
        "comprehensive data lineage, version traceability, and artifact integrity."
    ),
    verbose=False,
    allow_delegation=False,
    llm=governance_llm
)

# ==========================================
# 4. RUN EXTRACTION
# ==========================================
print("\n Starting Automated Extraction Node...")
model_metadata, feature_metadata = extract_model_artifacts(UPLOADED_PKL)
model_docs = extract_documentation(UPLOADED_DOCS)
print(" Extraction Complete! Handing data to Governance Agent...\n")

# ==========================================
# 5. TASK — output_pydantic enforces the contract
# ==========================================
governance_task = Task(
    description=(
        f"Conduct a rigorous governance audit on the provided model package.\n\n"
        f"INPUTS:\n"
        f"- Metadata: {model_metadata}\n"
        f"- Documentation: {model_docs}\n"
        f"- Feature Metadata: {feature_metadata}\n\n"
        f"CORE VALIDATION CHECKS TO PERFORM:\n"
        f"1. Documentation completeness & artifact integrity.\n"
        f"2. Explainability compliance per regulatory standards.\n"
        f"3. Data governance & lineage validation.\n"
        f"4. Model owner & version traceability.\n\n"
        f"SCORING RULES:\n"
        f"- Start at 100 points.\n"
        f"- Deduct 10-20 points for missing data lineage or feature descriptions.\n"
        f"- Deduct 25 points for missing model owner or version history.\n"
        f"- Deduct 30 points if explainability artifacts are entirely absent.\n\n"
        f"STATUS RULES:\n"
        f"- Score 85-100: PASS\n- Score 70-84: WARNING\n- Score < 70: FAIL"
    ),
    expected_output="A validated governance audit report matching the AgentReport schema.",
    agent=governance_agent,
    output_pydantic=AgentReport
)

crew = Crew(agents=[governance_agent], tasks=[governance_task], verbose=False)
result = await crew.kickoff_async()

# ==========================================
# 6. OUTPUT — validated object, no manual JSON parsing
# ==========================================
report = governance_task.output.pydantic

print("\n==========================================")
print("🎯 FINAL GOVERNANCE REPORT")
print("==========================================")
print(report.model_dump_json(indent=2))

os.makedirs("audit_outputs", exist_ok=True)
with open("audit_outputs/governance_report.json", "w") as f:
    json.dump(report.model_dump(), f, indent=2)

print("\n✅ Saved to audit_outputs/governance_report.json")

Upload your model (.pkl) file:


Saving loan_model_governance_package.pkl to loan_model_governance_package (1).pkl
Upload your documentation (.pdf/.md/.txt) file:


⚠️ No documentation uploaded — proceeding with placeholder.

🚀 Starting Automated Extraction Node...
🧠 Extracting metadata from: loan_model_governance_package (1).pkl...
📄 Extracting docs from: no_docs.txt...
✅ Extraction Complete! Handing data to Governance Agent...


🎯 FINAL GOVERNANCE REPORT
{
  "agent_name": "Lead AI Governance & Compliance Auditor",
  "score": 25,
  "status": "FAIL",
  "findings": [
    "No formal documentation was provided for the model, leading to significant gaps in artifact integrity and regulatory compliance.",
    "Absence of detailed feature metadata, including descriptions and data lineage, severely impacts data governance validation and traceability.",
    "No explainability reports or artifacts were provided, failing to meet regulatory standards for model transparency and interpretability (e.g., SR 11-7, EU AI Act).",
    "Model owner information is not explicitly provided, hindering traceability, accountability, and compliance with governance frameworks